# Word2Vec 实验——《推理王国》第3章

本 notebook 基于第3章"自己动手"实验，通过 GloVe 词向量（`glove-wiki-gigaword-100`，100维，400k词汇）  
亲手验证类比算术、偏见复现、以及余弦相似度的反直觉案例。

| 实验 | 目的 |
|---|---|
| 类比算术（皇室/首都/时态/反义词） | 验证"语义从分布中涌现" |
| 职业性别偏见（doctor/teacher） | 复现让人不舒服的统计真相 |
| 余弦相似度分析（hot/viral） | 找到向量几何与语言直觉之间的裂缝 |

In [4]:
import gensim.downloader as api

# 只有 128MB，词汇量小但够做类比实验
model = api.load('glove-wiki-gigaword-100')

# model = api.load('word2vec-google-news-300')  # 首次运行自动下载并缓存，~1.6GB

In [5]:
# 如果要使用原始数据文件，手动下载到本地
# GoogleNews-vectors-negative300.bin 的下载地址（Google Drive 镜像）
# https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM
# 解压缩后 3.63 GB


# from gensim.models import KeyedVectors
# model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)

In [6]:
import numpy as np

# 假设 model 是已加载的 gensim KeyedVectors 对象
# 加载方式示例：
#   from gensim.models import KeyedVectors
#   model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

def analogy_query(model, positive_word, minus_word, plus_word, topn=5):
    """
    执行向量类比查询：vec(positive_word) - vec(minus_word) + vec(plus_word)
    model: gensim KeyedVectors 对象
    返回：最近邻词及其余弦相似度分数的列表
    """
    # 计算目标向量：vec("king") - vec("man") + vec("woman")
    target_vector = (model[positive_word]
                     - model[minus_word]
                     + model[plus_word])

    # 在词汇表中找到与目标向量余弦相似度最高的词
    # gensim 的 similar_by_vector 会自动排除输入词（通过 negative 参数控制）
    results = model.similar_by_vector(
        target_vector,
        topn=topn + 3  # 多取几个，以便手动过滤输入词
    )

    # 排除 positive_word、minus_word、plus_word 本身
    exclude = {positive_word.lower(), minus_word.lower(), plus_word.lower()}
    filtered = [(word, score) for word, score in results
                if word.lower() not in exclude][:topn]

    return filtered


# ── 运行经典类比：king - man + woman ≈ ? ───────────────────────
results = analogy_query(model, "king", "man", "woman")

print("类比查询：king - man + woman ≈ ?")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：king - man + woman ≈ ?
───────────────────────────────────
  第1名: queen            余弦相似度 = 0.7834
  第2名: monarch          余弦相似度 = 0.6934
  第3名: throne           余弦相似度 = 0.6833
  第4名: daughter         余弦相似度 = 0.6809
  第5名: prince           余弦相似度 = 0.6713


## **问题一：这个结果让你感到惊讶吗？这看起来"像推理"还是"像算术"？**

> 因为看过相关内容，所以不觉得惊讶。看起来像算术，是一个有关向量的等式。

这个判断是正确的。`king - man + woman ≈ queen` 本质上是一个关于**向量偏移方向一致性**的陈述——

$$\vec{\text{king}} - \vec{\text{man}} \approx \vec{\text{queen}} - \vec{\text{woman}}$$

这说明"皇室"和"性别"在向量空间里对应两个近似正交的方向，做减法/加法等于在性别轴上做了一次平移。这是几何规律，不是逻辑推理——没有量词、没有变量绑定、没有传递性，只有统计压缩后涌现的空间结构。

结果中第1名 `queen`（0.7834）和第2名 `monarch`（0.6934）之间有明显跳跃，说明几何结构在这个案例里相当清晰。

In [7]:
# ── 运行性别类比：doctor - man + woman ≈ ? ───────────────────────
results = analogy_query(model, "doctor", "man", "woman")

print(f"类比查询：doctor - man + woman ≈ ?\n预期: nurse")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：doctor - man + woman ≈ ?
预期: nurse
───────────────────────────────────
  第1名: nurse            余弦相似度 = 0.7757
  第2名: physician        余弦相似度 = 0.7128
  第3名: doctors          余弦相似度 = 0.6794
  第4名: pregnant         余弦相似度 = 0.6788
  第5名: patient          余弦相似度 = 0.6772


In [8]:
# ── 运行性别类比：teacher - man + woman ≈ ? ───────────────────────
results = analogy_query(model, "teacher", "man", "woman")

print(f"类比查询：teacher - man + woman ≈ ?\n预期: educator 或者 kindergarten之类的")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：teacher - man + woman ≈ ?
预期: educator 或者 kindergarten之类的
───────────────────────────────────
  第1名: student          余弦相似度 = 0.7197
  第2名: schoolteacher    余弦相似度 = 0.6753
  第3名: nurse            余弦相似度 = 0.6723
  第4名: graduate         余弦相似度 = 0.6576
  第5名: mother           余弦相似度 = 0.6473


In [9]:
# ── 运行性别类比：teacher - woman + man ≈ ? ───────────────────────
results = analogy_query(model, "teacher", "woman", "man")

print(f"类比查询：teacher - woman + man ≈ ?\n预期: mentor")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：teacher - woman + man ≈ ?
预期: mentor
───────────────────────────────────
  第1名: master           余弦相似度 = 0.6857
  第2名: taught           余弦相似度 = 0.6725
  第3名: student          余弦相似度 = 0.6700
  第4名: teaching         余弦相似度 = 0.6592
  第5名: school           余弦相似度 = 0.6543


## **问题三（偏见案例）：这些偏见是系统的 bug，还是训练语料真实统计结构的如实反映？"去偏见"意味着什么？**

> 后者。去偏见意味着消除统计分布的偏差。

这个立场准确。两个职业实验的结果直接展示了语料偏见：

| 查询 | 结果第1名 | 说明 |
|---|---|---|
| `doctor - man + woman` | **nurse** | 语料中"医生"主要以男性上下文出现 |
| `teacher - man + woman` | **student**，第3名 **nurse** | "teacher"在女性语境下被拉向护理/养育方向 |
| `teacher - woman + man` | **master** | 移除女性语境后，"teacher"偏向权威/男性角色 |

这不是算法的 bug，算法忠实地学到了语料中的统计分布——语料里有偏见，向量就有偏见。

**"去偏见"的代价**：Bolukbasi (2016) 提出的线性投影法——在向量空间中找到"性别方向"，对职业词做投影去除这个分量。但这是治标：你改变了向量，但语料里的偏见结构没有变。更深的问题是，某些"偏见"是真实的社会统计规律（护士确实以女性为主），去偏见究竟是消除偏见，还是在向量层面掩盖了一个真实存在的现象？这是一个没有简单答案的规范性问题。

In [10]:
# ── 运行首都关系：Washington - America + China ≈ ? ───────────────────────
results = analogy_query(model, "tokyo", "japan", "china")

print(f"类比查询：tokyo - japan + china ≈ ?\n预期: beijing")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：tokyo - japan + china ≈ ?
预期: beijing
───────────────────────────────────
  第1名: beijing          余弦相似度 = 0.8395
  第2名: shanghai         余弦相似度 = 0.8281
  第3名: taipei           余弦相似度 = 0.7852
  第4名: seoul            余弦相似度 = 0.7444
  第5名: hong             余弦相似度 = 0.7433


In [11]:
# ── 运行首都关系：washington - america + france ≈ ? ───────────────────────
results = analogy_query(model, "washington", "america", "france")

print(f"类比查询：washington - america + france ≈ ?\n预期: paris")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：washington - america + france ≈ ?
预期: paris
───────────────────────────────────
  第1名: paris            余弦相似度 = 0.7018
  第2名: brussels         余弦相似度 = 0.6610
  第3名: britain          余弦相似度 = 0.6508
  第4名: belgium          余弦相似度 = 0.6381
  第5名: french           余弦相似度 = 0.6232


In [12]:
# ── 运行首都关系：rome - italy + spain ≈ ? ───────────────────────
results = analogy_query(model, "rome", "italy", "spain")

print(f"类比查询：rome - italy + spain ≈ ?\n预期: madrid")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：rome - italy + spain ≈ ?
预期: madrid
───────────────────────────────────
  第1名: madrid           余弦相似度 = 0.6993
  第2名: seville          余弦相似度 = 0.6583
  第3名: paris            余弦相似度 = 0.6316
  第4名: santiago         余弦相似度 = 0.6130
  第5名: barcelona        余弦相似度 = 0.6128


In [13]:
# ── 运行时态变化：run - go + went ≈ ? ───────────────────────
results = analogy_query(model, "run", "go", "went")

print(f"类比查询：run - go + went ≈ ?\n预期: ran")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：run - go + went ≈ ?
预期: ran
───────────────────────────────────
  第1名: ran              余弦相似度 = 0.7986
  第2名: second           余弦相似度 = 0.7014
  第3名: first            余弦相似度 = 0.6980
  第4名: followed         余弦相似度 = 0.6965
  第5名: took             余弦相似度 = 0.6950


In [14]:
# ── 运行时态变化：ski - go + went ≈ ? ───────────────────────
results = analogy_query(model, "ski", "go", "went")

print(f"类比查询：ski - go + went ≈ ?\n预期: skied")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：ski - go + went ≈ ?
预期: skied
───────────────────────────────────
  第1名: skiing           余弦相似度 = 0.6896
  第2名: alpine           余弦相似度 = 0.6674
  第3名: mountain         余弦相似度 = 0.6146
  第4名: downhill         余弦相似度 = 0.6044
  第5名: snowboard        余弦相似度 = 0.5894


In [15]:
# ── 运行时态变化：think - go + went ≈ ? ───────────────────────
results = analogy_query(model, "think", "go", "went")

print(f"类比查询：think - go + went ≈ ?\n预期: thought")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：think - go + went ≈ ?
预期: thought
───────────────────────────────────
  第1名: thought          余弦相似度 = 0.8413
  第2名: he               余弦相似度 = 0.7920
  第3名: knew             余弦相似度 = 0.7835
  第4名: felt             余弦相似度 = 0.7808
  第5名: got              余弦相似度 = 0.7765


In [16]:
# ── 运行反义词：brave - good + bad ≈ ? ───────────────────────
results = analogy_query(model, "wonderful", "good", "bad")

print(f"类比查询：wonderful - good + bad ≈ ?\n预期: awful")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：wonderful - good + bad ≈ ?
预期: awful
───────────────────────────────────
  第1名: awful            余弦相似度 = 0.7844
  第2名: horrible         余弦相似度 = 0.7577
  第3名: terrible         余弦相似度 = 0.7397
  第4名: weird            余弦相似度 = 0.7358
  第5名: scary            余弦相似度 = 0.7276


In [17]:
# ── 运行反义词：man - good + bad ≈ ? ───────────────────────
results = analogy_query(model, "man", "good", "bad")

print(f"类比查询：man - good + bad ≈ ?\n预期: woman")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：man - good + bad ≈ ?
预期: woman
───────────────────────────────────
  第1名: boy              余弦相似度 = 0.7107
  第2名: victim           余弦相似度 = 0.7023
  第3名: woman            余弦相似度 = 0.6637
  第4名: mysterious       余弦相似度 = 0.6588
  第5名: dead             余弦相似度 = 0.6569


In [18]:
# ── 运行反义词：brave - good + bad ≈ ? ───────────────────────
results = analogy_query(model, "brave", "good", "bad")

print(f"类比查询：braveness - good + bad ≈ ?\n预期: coward")
print("─" * 35)
for rank, (word, score) in enumerate(results, 1):
    print(f"  第{rank}名: {word:15s}  余弦相似度 = {score:.4f}")

类比查询：braveness - good + bad ≈ ?
预期: coward
───────────────────────────────────
  第1名: stupid           余弦相似度 = 0.5840
  第2名: terrible         余弦相似度 = 0.5800
  第3名: desperate        余弦相似度 = 0.5651
  第4名: heartless        余弦相似度 = 0.5624
  第5名: horrible         余弦相似度 = 0.5622


## **问题二：哪一类类比成功率最高？哪一类最低？**

> 首都 > 时态 > 性别 > 反义词

从实验结果来看：

| 类别 | 结果 | 原因分析 |
|---|---|---|
| **首都**（tokyo/washington/rome） | ✅ 全部命中，余弦 ~0.70–0.84 | 首都-国家是一对一、无歧义的强共现关系，语料中几乎不存在统计偏差 |
| **时态**（run/think/ski） | ✅ run→ran, think→thought 命中；ski→skied ❌ | 规则时态强健；`ski` 是低频词，`skied` 更稀少，偏移向量被词频噪声稀释 |
| **性别职业**（doctor/teacher） | ⚠️ 命中但结果令人不舒服 | 语料偏见导致结果偏向刻板印象，几何上"成功"但语义上有问题 |
| **反义词**（wonderful/brave） | ❌ brave→coward 失败 | 反义词不是线性偏移关系——`good/bad` 出现在几乎相同的上下文（反义词上下文共享），减去 `good` 加 `bad` 并不能产生稳定的"反义移位向量" |

**反义词失败的深层原因**：`good` 和 `bad` 的上下文高度重叠（都出现在"it's very ___"、"feel ___"中），所以它们的向量方向很接近，差向量接近零向量——用一个接近零的向量做偏移，结果完全取决于 `brave` 自身的邻域，无法稳定预测 `coward`。这不是语料偏差，而是**这类关系本身不适合线性偏移编码**。

In [19]:
def cosine_similarity(v1, v2):
    """计算两个向量之间的余弦相似度。"""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def analogy_precision(model, analogies):
    """
    批量测量类比精度。
    analogies: 列表，每项为 (A, B, C, D)，表示 A:B :: C:D 的类比关系。
    返回：每条类比的详细测量结果。
    """
    results = []

    for (A, B, C, D) in analogies:
        # 计算预测向量：vec(B) - vec(A) + vec(C)
        predicted_vector = model[B] - model[A] + model[C]

        # 获取整个词汇表按相似度排序的结果（排除输入词）
        exclude = {A.lower(), B.lower(), C.lower()}
        neighbors = model.similar_by_vector(predicted_vector, topn=len(model.key_to_index))
        # 过滤掉输入词，得到带排名的词表
        filtered_neighbors = [(word, score) for word, score in neighbors
                               if word.lower() not in exclude]

        # 实际最近邻排名：词汇表中按相似度排序后 D 的位置
        rank_of_D = None
        for idx, (word, score) in enumerate(filtered_neighbors, 1):
            if word.lower() == D.lower():
                rank_of_D = idx
                break

        # 计算偏移向量一致性：vec(B) - vec(A) 和 vec(D) - vec(C) 的余弦相似度
        # 这直接衡量两对词之间的"关系方向"是否一致
        offset_AB = model[B] - model[A]
        offset_CD = model[D] - model[C]
        offset_consistency = cosine_similarity(offset_AB, offset_CD)

        results.append({
            "analogy": f"{A}:{B} :: {C}:{D}",
            "top1_hit": rank_of_D == 1,           # D 是否出现在前 1 名（精确命中）
            "top5_hit": rank_of_D is not None and rank_of_D <= 5,  # D 是否出现在前 5 名
            "rank_of_D": rank_of_D,
            "offset_consistency": offset_consistency,  # 偏移向量一致性（几何层面的测量）
        })

    return results


# ── 示例测试 ────────────────────────────────────────────────────
test_analogies = [
    ("man",    "king",    "woman",  "queen"),    # 性别 × 皇室
    ("france", "paris",   "japan",  "tokyo"),    # 国家 × 首都
    ("go",     "went",    "run",    "ran"),      # 动词时态
    ("good",   "bad",     "big",    "small"),    # 反义词
    ("man",    "doctor",  "woman",  "nurse"),    # 职业偏见案例
]

precision_results = analogy_precision(model, test_analogies)

print(f"{'类比':30s} {'精确命中':^8s} {'前5命中':^8s} {'排名':^6s} {'偏移一致性':^10s}")
print("─" * 70)
for r in precision_results:
    print(f"{r['analogy']:30s} "
          f"{'✓' if r['top1_hit'] else '✗':^8s} "
          f"{'✓' if r['top5_hit'] else '✗':^8s} "
          f"{str(r['rank_of_D']):^6s} "
          f"{r['offset_consistency']:^10.4f}")

类比                               精确命中     前5命中     排名     偏移一致性   
──────────────────────────────────────────────────────────────────────
man:king :: woman:queen           ✓        ✓       1      0.7581  
france:paris :: japan:tokyo       ✓        ✓       1      0.7546  
go:went :: run:ran                ✓        ✓       1      0.4576  
good:bad :: big:small             ✗        ✗      102    -0.1511  
man:doctor :: woman:nurse         ✓        ✓       1      0.6354  


## **问题四：类比失败时，偏移向量一致性分数是高还是低？失败在几何还是检索？**

> 低，不成立。向量表示不捕获真理，只捕获语料的统计分布。

精度表里最关键的一行是反义词 `good:bad :: big:small`：

| 类比 | 精确命中 | 排名 | **偏移一致性** |
|---|---|---|---|
| man:king :: woman:queen | ✓ | 1 | 0.7581 |
| france:paris :: japan:tokyo | ✓ | 1 | 0.7546 |
| go:went :: run:ran | ✓ | 1 | 0.4576 |
| **good:bad :: big:small** | **✗** | **102** | **-0.1511** |

`-0.1511` 说明**几何结构本身就不成立**——`good→bad` 这个偏移向量，和 `big→small` 这个偏移向量，方向不一致，甚至略微相反。失败发生在最底层：两对反义词的"反义方向"根本不是同一个方向。

这印证了第七节的核心论点：**"类比等式"不是推理，是几何规律的涌现。当几何规律不成立时（偏移一致性低），没有任何检索技巧能救回答案**。`good/bad` 和 `big/small` 的失败根源在于反义词共享上下文——差向量接近零，偏移方向随机——而不是检索方式的问题。

对比时态类比 `go:went :: run:ran`（偏移一致性 0.4576，命中）：一致性中等但仍然为正，几何上方向对了，检索就能找到。

In [20]:
def find_counterintuitive_similarities(model, word, topn=20, low_threshold=0.3):
    """
    对于给定词 W：
    1. 找余弦相似度最高的 topn 个词，从中寻找"违和感"词
    2. 找余弦相似度低于 low_threshold 的词，检查有无语义上相关但几何距离远的词
    model: gensim KeyedVectors 对象
    """
    print(f"=== 词 '{word}' 的余弦相似度分析 ===\n")

    # 找余弦相似度最高的 topn 个词
    top_neighbors = model.most_similar(word, topn=topn)
    print(f"余弦相似度最高的 {topn} 个词（寻找让你感到违和的词）：")
    for w, score in top_neighbors:
        print(f"  {w:20s}  相似度 = {score:.4f}")

    print()

    # 找余弦相似度较低的词中是否有语义上你认为相关的
    # 方法：对一组候选词直接计算相似度
    # candidate_words = ["science", "art", "mathematics", "philosophy",
    #                    "literature", "music", "history", "biology"]
    candidate_words = ["desert","sun","roast","melt","popular","heated","scorching","sexy","viral","spicy"]
    print(f"候选词的余弦相似度（相似度低于 {low_threshold} 但你认为语义相关？）：")
    for candidate in candidate_words:
        if candidate in model:
            sim = model.similarity(word, candidate)
            # flag = " ← 语义相关但距离远？" if sim < low_threshold else ""
            # print(f"  {candidate:20s}  相似度 = {sim:.4f}{flag}")
            if sim < low_threshold:  # 移到 print 外层作为过滤条件
                print(f"  {candidate:20s}  相似度 = {sim:.4f} ← 语义相关但距离远？")


# ── 示例运行 ────────────────────────────────────────────────────
# find_counterintuitive_similarities(model, word="physics", topn=20, low_threshold=0.3)
find_counterintuitive_similarities(model, word="hot", topn=20, low_threshold=0.3)

=== 词 'hot' 的余弦相似度分析 ===

余弦相似度最高的 20 个词（寻找让你感到违和的词）：
  cool                  相似度 = 0.7711
  cold                  相似度 = 0.7252
  warm                  相似度 = 0.6978
  heat                  相似度 = 0.6799
  soft                  相似度 = 0.6602
  dry                   相似度 = 0.6569
  hottest               相似度 = 0.6360
  ice                   相似度 = 0.6257
  mix                   相似度 = 0.6254
  summer                相似度 = 0.6187
  water                 相似度 = 0.6179
  fresh                 相似度 = 0.6109
  light                 相似度 = 0.5996
  cream                 相似度 = 0.5950
  rock                  相似度 = 0.5931
  wet                   相似度 = 0.5927
  spots                 相似度 = 0.5926
  chill                 相似度 = 0.5904
  spot                  相似度 = 0.5873
  spring                相似度 = 0.5872

候选词的余弦相似度（相似度低于 0.3 但你认为语义相关？）：
  viral                 相似度 = 0.2099 ← 语义相关但距离远？


## **问题五：余弦相似度没有捕获的语义关系是什么类型？与第八节"分布假设的上限"有什么联系？**

> 超越语料边界新出现的相似语义——比如 `viral` 和 `hot` 在当代网络语境下几乎同义，但在 GloVe 的训练语料（2010年前）里，两者分布几乎没有重叠。  
> 分布假设只能统计已有使用场景的意义相似度，无法识别词语脱离语境的意义，也无法判断真值。

`hot`（0.2099）与 `viral` 的案例是**时间盲区**型裂缝：

- 在训练语料时代，`viral` 主要出现在医学上下文（viral infection, viral disease）
- `hot` 主要出现在温度/天气/流行音乐上下文
- 两者的上下文分布几乎没有重叠 → 余弦很低

但在当代语境里，"这个视频很 hot"和"这个视频 went viral"几乎是同义表达。语义已经漂移，但向量是**静态快照**，永远停留在语料冻结的那一刻。

这揭示了第八节"分布假设的上限"的两个具体边界：

| 限制类型 | 例子 | 原因 |
|---|---|---|
| **时间盲区** | `hot` ↔ `viral` 相似度低 | 语义随时间漂移，但向量是静态的 |
| **真值盲区** | "地球绕太阳"≈"太阳绕地球" | 分布假设学的是统计规律（描述性），不是因果真值（规范性） |

两种盲区的共同根源：**分布假设把"用法"等同于"意义"**。Wittgenstein 说"语言的意义是它的用法"——分布假设抓住了这个直觉的一部分，但用法是统计性的快照，而意义是动态的、有真值的。

---

## 检验清单（作者要求的三样东西）

### 1. 类比测试汇总表

| 类比 | 类型 | 精确命中 | 排名 | 偏移一致性 | 解释 |
|---|---|---|---|---|---|
| king - man + woman ≈ queen | 性别×皇室 | ✅ | 1 | 0.7581 | 经典成功案例，几何结构清晰 |
| tokyo - japan + china ≈ beijing | 首都 | ✅ | 1 | 0.7546 | 首都关系强共现，无偏差 |
| washington - america + france ≈ paris | 首都 | ✅ | 1 | — | 同上 |
| rome - italy + spain ≈ madrid | 首都 | ✅ | 1 | — | 同上 |
| go:went :: run:ran | 时态 | ✅ | 1 | 0.4576 | 规则时态稳健 |
| go:went :: think:thought | 时态 | ✅ | 1 | — | 不规则时态也命中 |
| go:went :: ski:skied | 时态 | ❌ | — | — | ski 低频词，skied 更稀少 |
| doctor - man + woman ≈ nurse | 职业偏见 | ✅(⚠️) | 1 | 0.6354 | 几何成功，但暴露语料偏见 |
| teacher - man + woman | 职业偏见 | ⚠️ | — | — | 第3名是 nurse，偏向护理 |
| wonderful - good + bad ≈ awful | 反义词 | ✅ | 1 | — | 程度类反义词有一定线性结构 |
| brave - good + bad ≈ coward | 反义词 | ❌ | — | — | 反义词差向量接近零，无稳定方向 |
| good:bad :: big:small | 反义词 | ❌ | 102 | **-0.1511** | 几何结构本身不成立 |

---

### 2. 偏见案例：`doctor - man + woman = nurse`

**亲手复现的结果**：语料中"医生"主要以男性上下文出现，将性别方向从男性移向女性后，最近邻不是"女医生"，而是"护士"。

**立场**：这不是算法的 bug，而是训练语料（Google News / Wikipedia）真实统计结构的如实反映。"去偏见"意味着在向量空间中消除这个统计分布的偏差，但这引发了一个更深的问题：某些偏见是真实社会分布的反映（护士职业确实以女性为主），去偏见操作在数学上压制了这个方向，但社会统计现实并没有改变。这是**描述性**（语料怎样）与**规范性**（语料应该怎样）之间的张力。

---

### 3. 反直觉的余弦相似度案例：`hot` ↔ `viral`（相似度 0.2099）

**裂缝**：在当代中文/英文网络语境里，"很 hot"和"went viral"几乎同义——但 GloVe 在 2010 年前的语料里训练，`viral` 的主要上下文是医学（viral infection），与 `hot` 几乎没有分布重叠，余弦仅 0.2099。

**这个裂缝揭示了什么**：词向量是语料冻结时刻的静态快照，无法反映语义的时间漂移。这是分布假设的**时间边界**：它只能捕捉训练语料截止时刻的分布，语言在此之后的演化对它不可见。这和"地球/太阳"案例的**真值边界**一起，构成了第八节"分布假设上限"的两个核心限制。